Dataset is about houses selling rate in Ames, Iowa between 2006 and 2010. Each row is a sold house and each column is a feature. Features describe everything about house and the column SelePrice is a price for transaction. My objective is predict price of new house based on the new estate's features.

In [1]:
import pandas as pd
from src.data_loader import data_loader

houses_raw = data_loader()

In [2]:
houses_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

We can see that we have in dataset null values. Now I am sure that I have to do with it something. I have three options:
1.  Drop columns with null values.
2.  Drop the rows with null values.
3.  Imput data with special sckit-learn method.

In [3]:
null_value = houses_raw.isnull().sum()
null_value = null_value[null_value > 0].sort_values(ascending=False)
null_value

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

We see which colums have null values. From previous info() function I know that most of them are string values. I have to know what null means.

In [4]:
houses_raw.sample(15)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
849,850,80,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,187000
102,103,90,RL,64.0,7018,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,6,2009,WD,Alloca,118964
325,326,45,RM,50.0,5000,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,87000
1387,1388,50,RM,60.0,8520,Pave,Grvl,Reg,Lvl,AllPub,...,0,NaN,GdWo,NaN,0,8,2007,CWD,Family,136000
1052,1053,60,RL,100.0,9500,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,NaN,0,6,2007,WD,Normal,165000
1231,1232,90,RL,70.0,7728,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdWo,NaN,0,5,2006,WD,Normal,132500
170,171,50,RM,NaN,12358,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,128500
577,578,80,RL,96.0,11777,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2006,WD,Abnorml,164500
163,164,45,RL,55.0,5500,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2007,WD,Normal,103200
1106,1107,20,RL,114.0,10357,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,179900


We know from description that null does not mean lack of value. For expample:
- PoolArea - NA means lack od pool,
- MiscFeature - NA means none,
- Alley - NA means no alley access,
- Fence - NA means no fence,
- MasVnrType - NA means none,
- FireplaceQu - NA means none,
- LotFrontage - lack of description,
- GarageType - NA means no garage,
- GarageFinish - NA means no garage,
- GarageYrBlt - NA means no garage.

We see corelation between garage features. Probably I will connect Na value into one feature or do something with it. Now I will see LotFrontage column's values.

In [9]:
lotFrontageColumn = pd.Series(houses_raw['LotFrontage'])
lotFrontageColumn.sample(10)

28      47.0
38      68.0
891     70.0
1230     NaN
914     30.0
188     64.0
688     60.0
1346     NaN
881     44.0
139     65.0
Name: LotFrontage, dtype: float64

From description we know that value represent linear feet of street connectecd to property. I assume that NA value means lack of measurement of conection, becasue every property has to have access to any street. Propaby I will imput minimal value from the feature to NA value.

In [10]:
lotFrontageColumn.min()

np.float64(21.0)